<a href="https://colab.research.google.com/github/rohitk1503/-Rohit-kumar-_BCS_secytasks_2026/blob/main/Copy_of_Rohitkumar_250917_week2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os

print("Connecting to Google Drive...")
drive.mount('/content/drive')
print("\n[SUCCESS] Google Drive successfully mounted!")

Connecting to Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

[SUCCESS] Google Drive successfully mounted!


In [ ]:
!pip install -q langchain-google-genai

from langchain_google_genai import ChatGoogleGenerativeAI

print("Configuring Gemini model...")
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
print("[SUCCESS] Gemini 2.5 Flash model successfully configured!")

Configuring Gemini model...
[SUCCESS] Gemini 2.5 Flash model successfully configured!


In [ ]:
import os
from langchain_core.documents import Document

print("Starting local chunking process...\n")

drive_root = '/content/drive/MyDrive'
cleaned_file_path = None

for root, dirs, files in os.walk(drive_root):
    if '.ipynb_checkpoints' in root: continue
    for file in files:
        if file.endswith('.txt'):
            cleaned_file_path = os.path.join(root, file)
            break
    if cleaned_file_path: break

if cleaned_file_path:
    print(f"[FOUND] Corpus file found at: {cleaned_file_path}")
    try:
        with open(cleaned_file_path, "r", encoding="utf-8") as f:
            full_text = f.read()

        raw_chunks = [p.strip() for p in full_text.split("\n\n") if p.strip()]

        if len(raw_chunks) <= 1:
            raw_chunks = [s.strip() + "." for s in full_text.split(".") if s.strip()]

        extracted_documents = []
        for i, chunk in enumerate(raw_chunks):
            doc = Document(
                page_content=chunk,
                metadata={
                    "source": os.path.basename(cleaned_file_path),
                    "hierarchy_title": f"Section_{i+1}",
                    "hierarchy_section": "Content"
                }
            )
            extracted_documents.append(doc)

        print(f"\n[SUCCESS] Local chunking complete! Total chunks extracted: {len(extracted_documents)}")
    except Exception as e:
        print(f"[ERROR] Failed to read the file: {e}")
else:
    print("[ERROR] No '.txt' file found in your Google Drive!")
    print("Please make sure you have uploaded the text file correctly.")

Starting local chunking process...

[FOUND] Corpus file found at: /content/drive/MyDrive/corpus.txt

[SUCCESS] Local chunking complete! Total chunks extracted: 7222


In [ ]:
!pip install -q reportlab

from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.colors import HexColor

pdf_output_path = '/content/context_document.pdf'

if cleaned_file_path and os.path.exists(cleaned_file_path):
    print("Starting PDF conversion...")

    with open(cleaned_file_path, 'r', encoding='utf-8') as f:
        text_content = f.read()

    doc = SimpleDocTemplate(pdf_output_path, pagesize=letter, rightMargin=40, leftMargin=40, topMargin=40, bottomMargin=40)
    story = []
    styles = getSampleStyleSheet()

    title_style = ParagraphStyle('Title', parent=styles['Heading1'], fontSize=22, textColor=HexColor('#1a365d'), spaceAfter=15)
    body_style = ParagraphStyle('Body', parent=styles['Normal'], fontSize=11, leading=16, textColor=HexColor('#2c3e50'), spaceAfter=12)

    story.append(Paragraph("Context Document Report", title_style))
    story.append(Spacer(1, 15))

    for para in text_content.split('\n'):
        if para.strip():
            story.append(Paragraph(para, body_style))

    doc.build(story)
    print(f"\n[SUCCESS] PDF generated successfully! You can download 'context_document.pdf' from the left sidebar panel.")
else:
    print("[ERROR] Please run Cell 3 first to locate the text file.")

Starting PDF conversion...

[SUCCESS] PDF generated successfully! You can download 'context_document.pdf' from the left sidebar panel.


In [ ]:
!pip install -q langchain langchain-community rank-bm25

from langchain_community.retrievers import BM25Retriever

print("Initializing retrieval engine...")
retriever = BM25Retriever.from_documents(extracted_documents)

user_query = input("Please enter your question: ")

if user_query.strip():
    print(f"\nProcessing query: {user_query}")
    retrieved_docs = retriever.invoke(user_query)

    context_text = "\n".join([doc.page_content for doc in retrieved_docs])

    prompt = f"""
    Answer the user's question based strictly on the provided context below.
    If the information is not in the context, say "I cannot find the answer in the provided document."

    Context:
    {context_text}

    Question: {user_query}
    Answer:
    """

    response = llm.invoke(prompt)

    print("\n[SUCCESS] Response generated successfully:")
    print("-" * 50)
    print(response.content)
    print("-" * 50)
else:
    print("[ERROR] Question cannot be empty. Please run the cell again and enter a query.")

Initializing retrieval engine...
Please enter your question: chief enemy?

Processing query: chief enemy?

[SUCCESS] Response generated successfully:
--------------------------------------------------
Caius Marcius is chief enemy to the people.
--------------------------------------------------
